Die benötigten Bibliotheken werden importiert

In [ ]:
import random
import simpy

Definierte Konstanten bzw. Variablen

In [ ]:
RANDOM_SEED = 42
TANKSTELLEN_GROEßE = 200            # Größe des Tankstellentanks (Liter)
SCHWELLENWERT = 25                  # minimales Level an der Tankstelle (% von max) 
AUTO_TANK_GROEßE = 50               # Größe des Autotanks (Liter)
AUTO_TANK_LEVEL = [5, 25]           # Min/max Level des Autotanks (Liter)
AUFFUELLGESCHWINDIGKEIT = 2         # Auffüllgeschwindigkeit des Kraftstoffes beim Autotank (Liter/ Sekunde)
TANKWAGEN_ZEIT = 300                # Zeit die eine Tankwagen zum ankommen benötigt (Sekunden)
ANKUFNTSINTERVALL_AUTO = [30, 300]  # Intervall zwischen den Ankuftszeiten der Autos [min, max] (Sekunden)
SIMULATIONS_ZEIT = 1000             # Simulationszeit (Sekunden)

Der Generator auto() wird definiert. 

In [ ]:
def auto(name: str, env: simpy.Environment, tankstelle: simpy.Resource, tankstellen_tank: simpy.Container):
    
    auto_tank_level = random.randint(*AUTO_TANK_LEVEL)
    print(f'{env.now:6.2f} s: {name} kommt an der Tankstelle an')  
    
    with tankstelle.request() as req:
        yield req

        benoetigter_kraftstoff = AUTO_TANK_GROEßE - auto_tank_level
        yield tankstellen_tank.get(benoetigter_kraftstoff)

        yield env.timeout(benoetigter_kraftstoff/ AUFFUELLGESCHWINDIGKEIT)

        print(f'{env.now:6.1f} s: {name} wurde vollgetankt mit {benoetigter_kraftstoff:.1f} L')

Der Generator tankstellen_steuerung() wird definiert. In der Funktion wird der Tank der Tankstelle nachgefüllt

In [ ]:
def tankstellen_steuerung(env: simpy.Environment, tankstellen_tank: simpy.Container):

    while True:
        if tankstellen_tank.level / tankstellen_tank.capacity * 100 < SCHWELLENWERT:
            
            print(f'{env.now:6.1f} s: Ruf den Tankwagen')
  
            yield env.process(tankwagen(env, tankstellen_tank))
        
        yield env.timeout(10) 

Der Generator tankwagen() wird definiert.

In [ ]:
def tankwagen(env: simpy.Environment, tankstellen_tank: simpy.Container):

    yield env.timeout(TANKWAGEN_ZEIT)
    nachfuellmenge = tankstellen_tank.capacity - tankstellen_tank.level     
    tankstellen_tank.put(nachfuellmenge)                                   

    print(f'{env.now:6.1f} s: Tankwagen ist angekommen und hat den Tankstellentank mit {nachfuellmenge:.1f} L nachgefüllt')

Der Generator auto_generieren() wird definiert.

In [ ]:
def auto_generieren(env: simpy.Environment, tankstelle: simpy.Resource, tankstellen_tank: simpy.Container):

    i = 0 
    while True:
        yield env.timeout(random.randint(*ANKUFNTSINTERVALL_AUTO))
        env.process(auto(f"Auto {i}", env, tankstelle, tankstellen_tank))
        i += 1

Einrichten und Start der Simulation

In [ ]:
print('Tankstelle nachfüllen')
random.seed(RANDOM_SEED)

Das Environment wird erstellt und die Prozesse werden gestartet

In [ ]:
env = simpy.Environment()
tankstelle = simpy.Resource(env, 2)
tankstellen_tank = simpy.Container(env, TANKSTELLEN_GROEßE, init = TANKSTELLEN_GROEßE)

env.process(tankstellen_steuerung(env, tankstellen_tank))
env.process(auto_generieren(env, tankstelle, tankstellen_tank))

Ausführen der Simulation

In [ ]:
env.run(until=SIMULATIONS_ZEIT)